# Continuous IQN Dead-End Detection — MIMIC-IV Analysis

Evaluates the trained **Continuous IQN** D- and R-networks on the MIMIC-IV sepsis cohort,
producing the same results, tables, and plots as the paper but for the continuous (2D action) model.

**Run this notebook in Google Colab.**  Add a secret named `GITHUB_TOKEN` with a personal
access token that has read access to the repo.

## Files sourced from the repo (`mimic` branch — already committed)

| File | Path |
|---|---|
| D-network checkpoint | `runs/iqn_continuous_mimic/best_q_parametersnegative.pt` |
| R-network checkpoint | `runs/iqn_continuous_mimic/best_q_parameterspositive.pt` |
| Config | `configs/iqn_continuous_mimic.yaml` |

## Files you must supply (large, not in repo)

| File | Path | How to get it |
|---|---|---|
| Test data | `data/continuous_mimic/rectilinear_processed/encoded_test.npz` | Step 7 of `train_mimic_continuous.ipynb` (`encoded_data.zip`) |
| Train data (norm stats) | `data/continuous_mimic/rectilinear_processed/encoded_train.npz` | Same zip |

> **After a full Colab training run**: re-download `best_q_parameters{negative,positive}.pt`
> from Step 7 of `train_mimic_continuous.ipynb` and commit them to the repo before running this notebook.

## Step 1: Clone repository and install dependencies

Add your GitHub personal access token as a Colab secret named `GITHUB_TOKEN`.
The repo (`mimic` branch) already contains:
- `runs/iqn_continuous_mimic/` — trained D- and R-network checkpoints
- `configs/iqn_continuous_mimic.yaml` — model hyperparameters

The large encoded-data files (`encoded_{test,train}.npz`) are **not** in the repo.
Run Step 2 to upload them, or generate them from scratch using
`train_mimic_continuous.ipynb` first.

In [ ]:
import os
from google.colab import userdata

# Clone only if the repo isn't already there (safe to re-run after restart)
if not os.path.exists('/content/tmp'):
    token = userdata.get('GITHUB_TOKEN')
    os.system(f'git clone -b mimic https://{token}@github.com/jaym-01/ContinuousDeD.git /content/tmp')

%pip install -r /content/tmp/requirements.txt -q

In [ ]:
# Restart the runtime so Python loads the pip-installed numpy/scipy from
# user site-packages instead of Colab's system versions (which can be broken).
# After the restart, run this notebook from the NEXT cell onwards.
import numpy as np
if np.__version__ != '2.4.2':
    print(f'numpy {np.__version__} detected — restarting to load 2.4.2 ...')
    import os; os.kill(os.getpid(), 9)
else:
    print(f'numpy {np.__version__} ✓ — no restart needed, continue to next cell')

In [ ]:
# Set working directory to the cloned repo
%cd /content/tmp

import os, sys
sys.path.insert(0, '/content/tmp')
print('Working directory:', os.getcwd())

## Step 2: Upload encoded data

The encoded data files are large and are not stored in the repo.
Run the cell below to upload `encoded_data.zip` (downloaded from Step 7 of
`train_mimic_continuous.ipynb`), or skip if you are re-running and the files
are already present in `data/continuous_mimic/rectilinear_processed/`.

In [ ]:
import os

DATA_DIR_CHECK = '/content/tmp/data/continuous_mimic/rectilinear_processed'
test_npz  = os.path.join(DATA_DIR_CHECK, 'encoded_test.npz')
train_npz = os.path.join(DATA_DIR_CHECK, 'encoded_train.npz')

if os.path.exists(test_npz) and os.path.exists(train_npz):
    print('Encoded data already present — skipping upload.')
else:
    from google.colab import files
    print('Upload encoded_data.zip (from train_mimic_continuous.ipynb Step 7)...')
    uploaded = files.upload()  # select encoded_data.zip
    for fname in uploaded:
        if fname.endswith('.zip'):
            os.makedirs(DATA_DIR_CHECK, exist_ok=True)
            os.system(f'unzip -o "{fname}" -d {DATA_DIR_CHECK}')
            print(f'Extracted {fname} to {DATA_DIR_CHECK}')

In [ ]:
# Cell 1 -- Imports and environment setup
import pickle, yaml, types
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from rl_utils import ContinuousIQN_OfflineAgent
from analysis_utils import pre_flag_splitting, create_analysis_df, compute_auc
from plot_utils import plot_value_hists

# base_analysis.py does `from thresholds import th`, but thresholds.py does
# not exist -- the `th` class lives in analysis_utils.py. Inject a shim module.
from analysis_utils import th as _th
_thresholds_mod = types.ModuleType('thresholds')
_thresholds_mod.th = _th
sys.modules['thresholds'] = _thresholds_mod
from base_analysis import comp_flag_agg_values

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Cell 2 -- Load config and encoded test data
REPO_ROOT   = os.getcwd()  # /content/tmp in Colab
CONFIG_PATH = os.path.join(REPO_ROOT, 'configs', 'iqn_continuous_mimic.yaml')
params = yaml.safe_load(open(CONFIG_PATH))

DATA_DIR = os.path.join(REPO_ROOT, params['data_dir'])
CKPT_DIR = os.path.join(REPO_ROOT, params['checkpoint_fname'])

print(f'Data dir : {DATA_DIR}')
print(f'Ckpt dir : {CKPT_DIR}')

enc_test = np.load(os.path.join(DATA_DIR, 'encoded_test.npz'), allow_pickle=True)
print(f"\nTest set : {len(enc_test['states'])} patients")
print(f"  states  shape : {enc_test['states'].shape}")
print(f"  actions shape : {enc_test['actions'].shape}")
print(f"  rewards shape : {enc_test['rewards'].shape}")

In [ ]:
# Cell 3 -- Compute per-dimension state normalisation stats from training data
#
# The continuous IQN normalises inputs using mean +/- 3*sigma bounds.
# Results are cached to CKPT_DIR/state_norm_stats.npz so re-runs are instant.

NORM_CACHE = os.path.join(CKPT_DIR, 'state_norm_stats.npz')

if os.path.exists(NORM_CACHE):
    nc = np.load(NORM_CACHE)
    state_mean = nc['state_mean']
    state_std  = nc['state_std']
    print(f'Loaded cached norm stats from {NORM_CACHE}')
else:
    print('Loading encoded_train.npz ...')
    enc_train = np.load(os.path.join(DATA_DIR, 'encoded_train.npz'), allow_pickle=True)

    # Vectorised masking -- avoids slow Python loop over 6419 trajectories
    states_full = enc_train['states']                           # (6419, 73, 64)
    lengths     = enc_train['lengths'].flatten().astype(int)    # (6419,)
    mask        = np.arange(states_full.shape[1])[None, :] < lengths[:, None]  # (6419, 73)
    all_states  = states_full[mask]                             # (total_valid_steps, 64)

    state_mean = all_states.mean(axis=0)             # (64,)
    state_std  = all_states.std(axis=0).clip(1e-8)   # (64,)

    np.savez(NORM_CACHE, state_mean=state_mean, state_std=state_std)
    print(f'Cached norm stats to {NORM_CACHE}')
    del enc_train, states_full, all_states

params['state_mean'] = state_mean
params['state_std']  = state_std
params['input_dim']  = state_mean.shape[0]  # 64

print(f'state_mean : min={state_mean.min():.4f}  max={state_mean.max():.4f}')
print(f'state_std  : min={state_std.min():.4f}  max={state_std.max():.4f}')

In [ ]:
# Cell 4 -- Load trained Continuous IQN models

def load_continuous_rl(params, sided_Q, device):
    """Load the best-epoch ContinuousIQN_OfflineAgent checkpoint."""
    ckpt_path = os.path.join(CKPT_DIR, f'best_q_parameters{sided_Q}.pt')
    # weights_only=False needed: checkpoint contains non-tensor objects
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

    agent = ContinuousIQN_OfflineAgent(
        state_size=params['input_dim'],
        params=params,
        sided_Q=sided_Q,
        device=device,
    )
    agent.network.load_state_dict(ckpt['rl_network_state_dict'])
    agent.eval()

    print(f"Loaded {sided_Q} network -- epoch {ckpt['epoch'] + 1}, "
          f"best val loss = {ckpt['validation_loss'][-1]:.6f}")
    return agent, ckpt


agent_dn, ckpt_dn = load_continuous_rl(params, 'negative', device)  # D-network
agent_rn, ckpt_rn = load_continuous_rl(params, 'positive', device)  # R-network

In [ ]:
# Cell 5 -- Training loss curves (Figure 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, ckpt, title, col in zip(
    axes,
    [ckpt_dn, ckpt_rn],
    ['D-Network (negative rewards)', 'R-Network (positive rewards)'],
    ['#2166ac', '#d6604d'],
):
    train_loss = ckpt['loss']
    val_loss   = ckpt['validation_loss']
    epochs_tr  = np.arange(1, len(train_loss) + 1)
    epochs_val = np.arange(1, len(val_loss)   + 1)

    ax.plot(epochs_tr,  train_loss, label='Train', color=col, linewidth=2)
    ax.plot(epochs_val, val_loss, label='Val', color=col, linewidth=2,
            linestyle='--', marker='o', markersize=4)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Normalised Quantile Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xticks(epochs_tr)

fig.suptitle('Continuous IQN Training Curves', fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

fig_loss = fig

In [ ]:
# Cell 6 -- Evaluate continuous Q-value distributions on the test set
#
# Adapts analysis_utils.get_dn_rn_info for the continuous IQN:
#   - Calls network(state, action, N) -> (T, N, 1) quantile samples
#   - Stores q_dn/q_rn per step as (N, 1) -- a single action slot
#   - Sets data['a'] = 0 (int) so pre_flag_splitting gather works unchanged
#
# Only trajectories with terminal reward +-1 are kept (censored -> skipped).

def get_continuous_dn_rn_info(agent_dn, agent_rn, encoded_data, device, num_q_samples=64):
    """Evaluate continuous IQN D/R-networks on all +-1 terminal trajectories.

    Returns a dict with the same schema as analysis_utils.get_dn_rn_info:
      traj, step, s, a, q_dn, q_rn, category, stay_id
    where q_dn / q_rn are (num_q_samples, 1) arrays per timestep.
    """
    agent_dn.eval()
    agent_rn.eval()

    data = {
        'traj': [], 'step': [], 's': [], 'a': [],
        'q_dn': [], 'q_rn': [], 'category': [], 'stay_id': [],
    }

    n_total = len(encoded_data['states'])
    n_kept  = 0

    with torch.no_grad():
        for traj in range(n_total):
            traj_len = int(encoded_data['lengths'][traj][0])
            traj_r   = encoded_data['rewards'][traj][:traj_len]  # (T, 1)

            # Terminal reward as scalar -- skip censored patients (reward = 0)
            terminal_r = float(traj_r[-1].flatten()[0])
            if terminal_r not in (-1.0, 1.0):
                continue
            category = -1 if terminal_r < 0 else 1

            traj_sid     = encoded_data['stay_ids'][traj]
            traj_states  = torch.from_numpy(
                encoded_data['states'][traj][:traj_len]
            ).float().to(device)   # (T, 64)
            traj_actions = torch.from_numpy(
                encoded_data['actions'][traj][:traj_len]
            ).float().to(device)   # (T, 2)

            # Full trajectory inference: (T, num_q_samples, 1)
            q_dn_dist, _ = agent_dn.network(traj_states, traj_actions, num_q_samples)
            q_rn_dist, _ = agent_rn.network(traj_states, traj_actions, num_q_samples)

            # Sort quantiles, clip to valid range, move to CPU numpy
            q_dn_np = np.sort(np.clip(q_dn_dist.cpu().numpy(), -1.0, 0.0), axis=1)  # (T, N, 1)
            q_rn_np = np.sort(np.clip(q_rn_dist.cpu().numpy(),  0.0, 1.0), axis=1)  # (T, N, 1)

            for i in range(traj_len):
                data['traj'].append(traj)
                data['step'].append(i)
                data['s'].append(traj_states[i].cpu().numpy())
                data['a'].append(0)           # single continuous action slot (index 0)
                data['stay_id'].append(traj_sid)
                data['q_dn'].append(q_dn_np[i])   # (N, 1)
                data['q_rn'].append(q_rn_np[i])   # (N, 1)
                data['category'].append(category)

            n_kept += 1

    print(f'Processed {n_kept}/{n_total} trajectories '
          f'({n_total - n_kept} censored patients skipped)')
    return data


CACHE_PATH = os.path.join(CKPT_DIR, 'value_data.pkl')

if os.path.exists(CACHE_PATH):
    print(f'Loading cached value data from {CACHE_PATH}')
    with open(CACHE_PATH, 'rb') as f:
        value_data = pickle.load(f)
else:
    print('Evaluating networks on test set...')
    value_data = get_continuous_dn_rn_info(
        agent_dn, agent_rn, enc_test, device, num_q_samples=64
    )
    with open(CACHE_PATH, 'wb') as f:
        pickle.dump(value_data, f)
    print(f'Cached to {CACHE_PATH}')

_cats = pd.DataFrame({'traj': value_data['traj'], 'cat': value_data['category']})\
         .drop_duplicates('traj')['cat']
print(f"Survivors     (cat= 1): {(_cats == 1).sum()}")
print(f"Non-survivors (cat=-1): {(_cats ==-1).sum()}")

In [ ]:
# Cell 7 -- Pre-flag splitting and analysis DataFrames
#
# pre_flag_splitting is reused unchanged. The key compatibility trick:
#   q_dn / q_rn stored as (N, 1) per step -> stacked to (T, N, 1) per traj
#   action index = 0 throughout -> gather(dim=2, index=0) selects the one slot
#   result: selected-action CVaR shape (T, 20) and median-V shape (T, 20)

VaR_thresholds = np.round(np.linspace(0.05, 1.0, num=20), decimals=2)
print(f'VaR thresholds (alpha): {VaR_thresholds}')

print('\nRunning pre_flag_splitting...')
results = pre_flag_splitting(value_data, VaR_thresholds, distributional=True)

n_survivors    = len(results['survivors']['dn_q_selected_action_traj'])
n_nonsurvivors = len(results['nonsurvivors']['dn_q_selected_action_traj'])
print(f'\nSurvivors: {n_survivors}, Non-survivors: {n_nonsurvivors}')

surv_df, nonsurv_df = create_analysis_df(results, n_survivors, n_nonsurvivors)

print(f'surv_df    shape: {surv_df.shape}')
print(f'nonsurv_df shape: {nonsurv_df.shape}')
print(f"\nTime steps (survivors)    : {sorted(surv_df['step'].unique())[:8]} ...")
print(f"Time steps (non-survivors): {sorted(nonsurv_df['step'].unique())[:8]} ...")

In [ ]:
# Cell 8 -- Value distribution histograms (Figure 2)

shared_steps = sorted(
    set(surv_df['step'].unique()) & set(nonsurv_df['step'].unique()),
    reverse=True,
)
print(f'Shared time steps available: {shared_steps}')

candidate_steps = [-1, -4, -8, -12, -24]
plot_steps = [s for s in candidate_steps if s in shared_steps]
if not plot_steps:
    plot_steps = shared_steps[:min(3, len(shared_steps))]
print(f'Plotting steps: {plot_steps}')

fig_hists = []
for step_num in plot_steps:
    fig, axs = plt.subplots(1, 4, figsize=(20, 4))
    var_idx = 9  # alpha = 0.50
    plot_value_hists(axs, nonsurv_df, surv_df, step_num, var_idx=var_idx)
    fig.suptitle(
        f'Value Distributions at step {step_num}h from end  '
        f'(alpha = {VaR_thresholds[var_idx]:.2f})',
        fontsize=12,
    )
    axs[0].legend(fontsize=8)
    fig.tight_layout()
    plt.show()
    fig_hists.append(fig)

In [ ]:
# Cell 9 -- ROC curves and AUC table (Figure 3)

print('Computing AUC...')
fpr, tpr, auc_out = compute_auc(
    surv_df, nonsurv_df, n_survivors, n_nonsurvivors, iqn_size=20
)
auc_vals = auc_out[0]  # shape (20,)

print('\nAUC per CVaR alpha:')
print(f'{"Alpha":>8}  {"AUC":>8}')
print('-' * 20)
for alpha, auc_v in zip(VaR_thresholds, auc_vals):
    print(f'{alpha:>8.2f}  {auc_v:>8.4f}')
print(f'\nBest AUC : {auc_vals.max():.4f}  (alpha = {VaR_thresholds[auc_vals.argmax()]:.2f})')
print(f'Mean AUC : {auc_vals.mean():.4f}')

ns_colors = sns.color_palette('Blues_r', n_colors=20)
fig, ax = plt.subplots(figsize=(7, 6))
for ii in range(20):
    label = (f'alpha={VaR_thresholds[ii]:.2f} (AUC={auc_vals[ii]:.3f})'
             if ii % 4 == 0 else None)
    ax.plot(fpr[:, ii], tpr[:, ii], color=ns_colors[ii],
            linewidth=1, alpha=0.8, label=label)

ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='Random')
ax.set_xlabel('False Positive Rate (survivors flagged)')
ax.set_ylabel('True Positive Rate (non-survivors flagged)')
ax.set_title('ROC Curves -- Continuous IQN DeD\n(each curve = one CVaR alpha)')
ax.legend(fontsize=8, loc='lower right')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

fig_roc = fig

In [ ]:
# Cell 10 -- Aggregated value traces around first flag (Figure 4)

print('Computing aggregated value traces...')
ns_vals = comp_flag_agg_values(nonsurv_df, distributional=True)
s_vals  = comp_flag_agg_values(surv_df,    distributional=True)

print(f'Non-survivor flag-aligned rows : {len(ns_vals)}')
print(f'Survivor flag-aligned rows     : {len(s_vals)}')

if len(ns_vals) == 0 and len(s_vals) == 0:
    print('\nNo trajectories raised a flag with the current checkpoint.')
    print('Re-run from Cell 6 after downloading fully-trained checkpoints from Colab.')
    fig_traces = None
else:
    ns_colors = sns.color_palette('Blues',  n_colors=20)
    s_colors  = sns.color_palette('Greens', n_colors=20)
    titles = [r'$Q_{D}$', r'$Q_{R}$', r'$V_{D}$', r'$V_{R}$']
    window_pre, window_post = 12, 8

    fig, axs = plt.subplots(2, 2, figsize=(12, 8), sharex=True)

    for idx, item in enumerate(['q_dn', 'q_rn', 'v_dn', 'v_rn']):
        ax = axs[idx // 2, idx % 2]
        for ivar in range(20):
            if len(ns_vals) > 0:
                ns_temp = ns_vals[ns_vals.var_index == ivar]
                if len(ns_temp) > 0:
                    sns.lineplot(x=ns_temp['step'], y=ns_temp[item],
                                 color=ns_colors[ivar], errorbar=None, linewidth=1, ax=ax)
            if len(s_vals) > 0:
                s_temp = s_vals[s_vals.var_index == ivar]
                if len(s_temp) > 0:
                    sns.lineplot(x=s_temp['step'], y=s_temp[item],
                                 color=s_colors[ivar], errorbar=None, linewidth=1, ax=ax)
        ax.axvline(x=0, ls='--', color='gray', alpha=0.5)
        ax.set_title(titles[idx], fontsize=13)
        ax.set_ylabel('')
        ax.set_xticks(np.arange(-window_pre, window_post + 1))
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

    from matplotlib.lines import Line2D
    fig.legend(
        handles=[
            Line2D([0], [0], color=ns_colors[9], linewidth=2, label='Non-survivors'),
            Line2D([0], [0], color=s_colors[9],  linewidth=2, label='Survivors'),
            Line2D([0], [0], color='gray', ls='--', linewidth=1, label='First flag'),
        ],
        loc='upper right', fontsize=9,
    )
    fig.suptitle('Value Traces Around First DeD Flag -- Continuous IQN', fontsize=13)
    fig.tight_layout()
    plt.show()
    fig_traces = fig

In [ ]:
# Cell 11 -- Summary statistics

from analysis_utils import compare_flag_range

best_alpha_idx = int(auc_vals.argmax())
best_alpha     = float(VaR_thresholds[best_alpha_idx])
best_auc       = float(auc_vals[best_alpha_idx])

first_flags_ns, first_flags_s = [], []

for traj in range(n_nonsurvivors):
    dt = nonsurv_df[nonsurv_df.traj == traj]
    flags_all = np.stack(dt.apply(compare_flag_range, axis=1).values)  # (T, 1001, 20)
    any_flag  = np.any(flags_all[:, :, best_alpha_idx], axis=1)         # (T,)
    if any_flag.any():
        first_flags_ns.append(int(np.where(any_flag)[0][0]) + int(dt['step'].iloc[0]))

for traj in range(n_survivors):
    dt = surv_df[surv_df.traj == traj]
    flags_all = np.stack(dt.apply(compare_flag_range, axis=1).values)
    any_flag  = np.any(flags_all[:, :, best_alpha_idx], axis=1)
    if any_flag.any():
        first_flags_s.append(int(np.where(any_flag)[0][0]) + int(dt['step'].iloc[0]))

pct_ns = 100 * len(first_flags_ns) / max(n_nonsurvivors, 1)
pct_s  = 100 * len(first_flags_s)  / max(n_survivors,    1)

print('=' * 56)
print(' Continuous IQN Dead-End Detection -- Summary')
print('=' * 56)
print(f' Total test trajectories   : {n_survivors + n_nonsurvivors}')
print(f'   Survivors               : {n_survivors}')
print(f'   Non-survivors           : {n_nonsurvivors}')
print(f' Best CVaR alpha           : {best_alpha:.2f}  (index {best_alpha_idx})')
print(f' Best AUC                  : {best_auc:.4f}')
print(f' Mean AUC across alphas    : {auc_vals.mean():.4f}')
print(f' Non-survivors flagged     : {len(first_flags_ns)}/{n_nonsurvivors} ({pct_ns:.1f}%)')
print(f' Survivors flagged (FP)    : {len(first_flags_s)}/{n_survivors} ({pct_s:.1f}%)')
if first_flags_ns:
    print(f' Mean first-flag step (NS) : {np.mean(first_flags_ns):.1f}h from end')
print('=' * 56)

In [ ]:
# Cell 12 -- Save all figures to runs/iqn_continuous_mimic/

os.makedirs(CKPT_DIR, exist_ok=True)

fig_loss.savefig(
    os.path.join(CKPT_DIR, 'fig1_training_curves.png'), dpi=150, bbox_inches='tight')
print('Saved: fig1_training_curves.png')

for i, fh in enumerate(fig_hists):
    fh.savefig(
        os.path.join(CKPT_DIR, f'fig2_hist_step{plot_steps[i]}.png'),
        dpi=150, bbox_inches='tight')
    print(f'Saved: fig2_hist_step{plot_steps[i]}.png')

fig_roc.savefig(
    os.path.join(CKPT_DIR, 'fig3_roc_curves.png'), dpi=150, bbox_inches='tight')
print('Saved: fig3_roc_curves.png')

if fig_traces is not None:
    fig_traces.savefig(
        os.path.join(CKPT_DIR, 'fig4_value_traces.png'), dpi=150, bbox_inches='tight')
    print('Saved: fig4_value_traces.png')
else:
    print('fig4_value_traces.png skipped (no flags raised yet)')

auc_table = pd.DataFrame({'alpha': VaR_thresholds, 'auc': auc_vals})
auc_table.to_csv(os.path.join(CKPT_DIR, 'auc_table.csv'), index=False)
print('Saved: auc_table.csv')

print(f'\nAll outputs saved to: {CKPT_DIR}')